In [ ]:
!pip install transformers datasets seqeval accelerate evaluate

In [ ]:
from datasets import Dataset, load_dataset
from evaluate import load
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate
import numpy as np

In [ ]:
datasets = load_dataset("json", data_files="/content/ner_bio_3.json")

In [ ]:
split_dataset = datasets["train"].train_test_split(test_size=0.2, seed=42)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

In [ ]:
labels = sorted(list({label for row in train_dataset for label in row["ner_tags"]}))
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

In [ ]:
print("Labels:", labels)

In [ ]:
model_name = "xlm-roberta-base"  # hoặc "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    all_labels = []
    for i, labels_per_example in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            else:
                label_ids.append(label2id[labels_per_example[word_idx]])
        all_labels.append(label_ids)
    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

tokenized_datasets = datasets.map(tokenize_and_align_labels, batched=True, remove_columns=datasets["train"].column_names)

In [ ]:
tokenized_datasets_test = test_dataset.map(tokenize_and_align_labels, batched=True, remove_columns=test_dataset.column_names)

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [ ]:
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = [[id2label[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./ner_contract_model",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    report_to="none",
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_strategy="epoch",
    fp16=True,  # Bật True nếu bạn dùng GPU hỗ trợ
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
results = trainer.evaluate(tokenized_datasets_test)
print(results)

In [ ]:
from transformers import pipeline
ner = pipeline("ner", model=model, tokenizer="xlm-roberta-base", aggregation_strategy="simple")
text = """
BÊN CHO THUÊ ĐẤT ỦY BAN NHÂN DÂN QUẬN 7 Địa chỉ Số 7, đường Tân Phú, Phường Tân Phú, Quận 7, TP. Hồ Chí Minh . Đại diện Ông Nguyễn Văn Khánh – Chủ tịch Ủy ban nhân dân Quận 7.
"""
print(ner(text))

In [ ]:
ner_results = ner(text)
grouped_entities = []
current_group = None

for item in ner_results:
    entity_group = item['entity_group']
    word = item['word'].strip() # Loại bỏ khoảng trắng thừa nếu có

    # Nếu đây là thực thể đầu tiên hoặc thực thể này khác loại
    # hoặc bị cách biệt (start position không tiếp nối end position của thực thể trước)
    if (current_group is None or
        entity_group != current_group['entity_group'] or
        item['start'] > current_group['end']):

        # Lưu lại thực thể đã hoàn thành (nếu có)
        if current_group is not None:
            grouped_entities.append(current_group)

        # Bắt đầu một nhóm mới
        current_group = {
            'entity_group': entity_group,
            'word': word,
            'start': item['start'],
            'end': item['end']
        }
    else:
        # Nếu cùng nhóm và tiếp nối, thì gộp từ và cập nhật vị trí kết thúc
        current_group['word'] += word
        current_group['end'] = item['end']

# Thêm nhóm cuối cùng sau khi lặp xong
if current_group is not None:
    grouped_entities.append(current_group)

# -----------------------------------------------
# ĐỊNH DẠNG KẾT QUẢ CUỐI CÙNG
# -----------------------------------------------

final_results = []
for entity in grouped_entities:
    final_results.append({
        'entity_group': entity['entity_group'],
        'word': entity['word'].replace('##', '') # Loại bỏ ký tự sub-word nếu có
    })

# In ra kết quả đã gộp
print(final_results)

In [ ]:
# trainer.save_model("./ner_contract_model")